# Qdrant Collection Setup

This notebook creates and populates the Qdrant vector collection used by the retrieval pipeline. It validates chunk-to-embedding alignment, generates deterministic point IDs, attaches legal metadata as payloads, and verifies the indexed collection.

Qdrant is expected to be available locally at `http://localhost:6333`.


## 1. Import the required libraries

Load JSON, deterministic UUID, path, and Qdrant client utilities.


In [10]:
from pathlib import Path
import json
import uuid

from qdrant_client import QdrantClient, models

## 2. Define files and collection settings

Locate chunks and embeddings and configure the local Qdrant collection.


In [2]:
# Run this notebook from notebooks/ so the project root is its parent.
PROJECT_ROOT = Path.cwd().parent

CHUNKS_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "crmp_chunks.json"
)

EMBEDDINGS_FILE = (
    PROJECT_ROOT
    / "data"
    / "embeddings"
    / "crmp_bge_m3_embeddings.json"
)

QDRANT_URL = "http://localhost:6333"

COLLECTION_NAME = "crmp_bge_m3"


## 3. Load chunks and embeddings

Read both datasets and display their core metadata.


In [3]:
with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    chunks_data = json.load(f)

with open(EMBEDDINGS_FILE, "r", encoding="utf-8") as f:
    embeddings_data = json.load(f)

chunks = chunks_data["chunks"]
embeddings = embeddings_data["embeddings"]

print(f"Chunks: {len(chunks)}")
print(f"Embeddings: {len(embeddings)}")
print(f"Modelo: {embeddings_data['embedding_model']}")
print(f"Dimensão: {embeddings_data['embedding_dimension']}")

Chunks: 1617
Embeddings: 1617
Modelo: bge-m3
Dimensão: 1024


## 4. Validate dataset lengths

Confirm that every chunk has one corresponding embedding.


In [4]:
assert len(chunks) == len(embeddings)

## 5. Validate identifier alignment

Ensure the chunk and embedding datasets contain exactly the same IDs.


In [5]:
chunk_ids = {chunk["id"] for chunk in chunks}
embedding_ids = {item["id"] for item in embeddings}

# Prevent vectors from being attached to the wrong legal text.
assert chunk_ids == embedding_ids

print("Chunks e embeddings correspondem.")


Chunks e embeddings correspondem.


## 6. Connect to Qdrant

Create the client and inspect the collections available on the local service.


In [6]:
client = QdrantClient(
    url=QDRANT_URL
)

print(client.get_collections())

collections=[]


## 7. Determine vector dimensions

Read the vector size directly from the embedding metadata.


In [7]:
VECTOR_SIZE = embeddings_data["embedding_dimension"]

print(VECTOR_SIZE)

1024


## 8. Create the collection when needed

Initialize a cosine-distance collection without overwriting an existing one.


In [8]:
# Preserve an existing collection so reruns do not delete indexed data.
if not client.collection_exists(COLLECTION_NAME):

    client.create_collection(
        collection_name=COLLECTION_NAME,

        vectors_config=models.VectorParams(
            size=VECTOR_SIZE,
            distance=models.Distance.COSINE
        )
    )

    print(
        f"Collection criada: "
        f"{COLLECTION_NAME}"
    )

else:

    print(
        f"Collection já existe: "
        f"{COLLECTION_NAME}"
    )


Collection criada: crmp_bge_m3


## 9. Index records by chunk ID

Build dictionaries for efficient and explicit chunk-to-vector joins.


In [9]:
chunks_by_id = {
    chunk["id"]: chunk
    for chunk in chunks
}

embeddings_by_id = {
    item["id"]: item
    for item in embeddings
}

## 10. Define deterministic Qdrant IDs

Map each readable chunk ID to a stable UUID accepted by Qdrant.


In [11]:
def qdrant_id(chunk_id):
    return str(
        # UUID5 makes the Qdrant point ID stable across repeated indexing runs.
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            f"municipal-rag-lab:{chunk_id}"
        )
    )


## 11. Test point ID generation

Confirm that a representative chunk produces a valid deterministic UUID.


In [12]:
print(
    qdrant_id(
        "crmp_a_1_1_chunk_001"
    )
)

71b1b930-e960-54d4-9fc8-c7a8fc7bf1cf


## 12. Build Qdrant points

Combine each vector with legal hierarchy, source references, and chunk text.


In [13]:
points = []

for chunk_id, chunk in chunks_by_id.items():

    embedding = embeddings_by_id[chunk_id]

    # Store citation and hierarchy fields alongside each vector for retrieval output.
    payload = {
        "chunk_id": chunk["id"],

        "article_id": chunk["article_id"],
        "article": chunk["article"],
        "article_title": chunk["article_title"],

        "part": chunk["part"],
        "title": chunk["title"],
        "chapter": chunk["chapter"],

        "page_start": chunk["page_start"],
        "page_end": chunk["page_end"],

        "chunk_index": chunk["chunk_index"],
        "num_chunks": chunk["num_chunks"],

        "embedding_model": embeddings_data["embedding_model"],

        "text": chunk["text"]
    }

    points.append(
        models.PointStruct(
            id=qdrant_id(chunk_id),
            vector=embedding["embedding"],
            payload=payload
        )
    )

print(f"Pontos preparados: {len(points)}")


Pontos preparados: 1280


## 13. Test a full upsert

Upload all points in a single operation as a simple indexing path.


In [14]:
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
    wait=True
)

print("Pontos inseridos.")

Pontos inseridos.


## 14. Upsert in batches

Upload points in smaller groups for a more scalable and observable indexing process.


In [15]:
# Batching limits request size and provides visible indexing progress.
BATCH_SIZE = 100

for start in range(
    0,
    len(points),
    BATCH_SIZE
):

    batch = points[
        start:start + BATCH_SIZE
    ]

    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch,
        wait=True
    )

    print(
        f"Inseridos "
        f"{min(start + BATCH_SIZE, len(points))}"
        f"/{len(points)}"
    )


Inseridos 100/1280
Inseridos 200/1280
Inseridos 300/1280
Inseridos 400/1280
Inseridos 500/1280
Inseridos 600/1280
Inseridos 700/1280
Inseridos 800/1280
Inseridos 900/1280
Inseridos 1000/1280
Inseridos 1100/1280
Inseridos 1200/1280
Inseridos 1280/1280


## 15. Inspect collection metadata

Retrieve the final collection configuration and status.


In [16]:
info = client.get_collection(
    COLLECTION_NAME
)

print(info)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=1280 segments_count=8 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, pre

## 16. Verify the indexed point count

Compare the number of stored points with the expected dataset size.


In [17]:
print(
    "Points:",
    info.points_count
)

Points: 1280


## 17. Display the expected record count

Return the number of source chunks for a final manual comparison.


In [18]:
len(chunks)

1617